In [ ]:
import subprocess, sys, os

"""
This output is:
 1. pubs.txt - Paper list for CWTS tool
    Columns: int_id  paper identifier for cwts, core_pub (always 1 as not using core feature, but it has to be in)

 2. cit_links.txt - Citation network edges
    Columns: 
    int_id1 - citing paper, 
    int_id2 - cited paper, 
    weight - citation strength 0-2 higher= stronger
    Note: Each edge appears twice (A→B and B→A) for undirected format
        paper 5 cites Paper 12  →  row: 5, 12, 0.85
        Paper 12 cites Paper 5  →  row: 12, 5, 0.85  (same edge, reversed)

 3. pub_metadata.txt - Paper details lookup table
    Columns: int_id, pub_id, is_frontiers, journal, date, title
    - int_id: sequential CWTS ID (joins to classification.txt)
    - pub_id: airak PublicationId (joins to BigQuery tables)
 JOIN KEY: int_id links all files together, this is cwts identifier
"""
print("ere")
env = os.environ.copy()
print("ere")
env["START_YEAR"] = "2023"
env["END_YEAR"] = "2026"
env["NETWORK_MODE"] = "full"
env["RUN_TIMESTAMP"] = run_timestamp  # Must be uppercase to match cwts_export.py
print("ere")
result = subprocess.run(
    [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
)
print("ere")
print(result.stdout)
print(result.stderr)
print("last")

ere
ere
ere
ere

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026-07-08 14:41:31,103 [INFO] ============================================================
2026-07-08 14:41:31,103 [INFO] Configuration
2026-07-08 14:41:31,103 [INFO] ============================================================
2026-07-08 14:41:31,103 [INFO]   BQ_PROJECT              : ocean-tech-adv-analytics-p-usr
2026-07-08 14:41:31,103 [INFO]   AIRAK_DATASET           : ocean-breeze-tier-1.airak
2026-07-08 14:41:31,103 [INFO]   NETWORK_MODE            : full
2026-07-08 14:41:31,103 [INFO]   START_YEAR              : 2023
2026-07-08 14:41:31,103 [INFO]   END_YEAR                : 2026
2026-07-08 14:41:31,104 [INFO]   TOP_N_JOURNALS          : 5
2026-07-08 14:41:31,104 [INFO]   JOURNAL_IDS_OVERRIDE    : (none)
2026-07-08 14:41:31,

In [ ]:
import subprocess
import datetime
import os
import pandas as pd

# --- Parameters ---
params = {
    "largest_component_only": "true",
    "iterations": "100",
    "micro_resolution": "5e-4",
    "micro_min_cluster_size": "1000",
    "meso_resolution": "5e-6",
    "meso_min_cluster_size": "5000",
    "macro_resolution": "1e-6",
    "macro_min_cluster_size": "20000",
}

input_files = {
    "pubs": "cwts_output/pubs.txt",
    "cit_links": "cwts_output/cit_links.txt",
    "output": "cwts_output/classification.txt",
    "jar": "publicationclassification.jar",
}


result = subprocess.run(
    [
        "java",
        "-cp",
        input_files["jar"],
        "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
        input_files["pubs"],
        input_files["cit_links"],
        input_files["output"],
        params["largest_component_only"],
        params["iterations"],
        params["micro_resolution"],
        params["micro_min_cluster_size"],
        params["meso_resolution"],
        params["meso_min_cluster_size"],
        params["macro_resolution"],
        params["macro_min_cluster_size"],
    ],
    capture_output=True,
    text=True,
)

# --- Log ---
os.makedirs("logs", exist_ok=True)
log_path = f"logs/cwts_run_{run_timestamp}.log"

with open(log_path, "w") as f:
    f.write(f"CWTS Publication Classification Run\n")
    f.write(f"{'='*50}\n")
    f.write(f"Timestamp : {run_timestamp}\n\n")

    f.write(f"Input Files\n{'-'*30}\n")
    for k, v in input_files.items():
        f.write(f"  {k:<20}: {v}\n")

    f.write(f"\nParameters\n{'-'*30}\n")
    for k, v in params.items():
        f.write(f"  {k:<26}: {v}\n")

    f.write(f"\nReturn Code: {result.returncode}\n")

    f.write(f"\nSTDOUT\n{'-'*30}\n")
    f.write(result.stdout or "(empty)\n")

    f.write(f"\nSTDERR\n{'-'*30}\n")
    f.write(result.stderr or "(empty)\n")

print(f"Log written to: {log_path}")
print(result.stdout)
if result.stderr:
    print(result.stderr)

# Load classification.txt
classification = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["int_id", "micro", "meso", "macro"],
)

# Upload to BigQuery
BQ_DEST_PROJECT = "ocean-tech-adv-analytics-c-tfs"
BQ_DEST_DATASET = "scope_drift_raw"


# classification.to_gbq(
#     f"{dataset}.classification_raw_{run_timestamp}",
#     project_id=project,
#     if_exists="replace",
# )

# Upload to BigQuery

classification.to_gbq(
    f"{BQ_DEST_DATASET}.classification_raw_{run_timestamp}",
    project_id=BQ_DEST_PROJECT,
    if_exists="replace",
)
print(f"  → BigQuery: {BQ_DEST_DATASET}.classification_raw_{run_timestamp}")

In [ ]:
import pandas as pd


df = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)


print(f"Total classified: {len(df):,}")


for level in ["micro", "meso", "macro"]:

    vc = df[level].value_counts()

    print(f"\n{level.upper()}: {len(vc):,} clusters")

    print(f"  Largest : {vc.iloc[0]:,} ({vc.iloc[0]/len(df)*100:.1f}%)")

    print(f"  Smallest: {vc.iloc[-1]:,}")

    print(f"  Median  : {vc.median():.0f}")

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Total classified: 3,844,339

MICRO: 1,818 clusters
  Largest : 16,117 (0.4%)
  Smallest: 1,000
  Median  : 1772

MESO: 183 clusters
  Largest : 298,499 (7.8%)
  Smallest: 5,076
  Median  : 12444

MACRO: 24 clusters
  Largest : 1,237,462 (32.2%)
  Smallest: 20,510
  Median  : 92906


### Labelling with GPT

In [ ]:
# timestamp = "20260710_144411" # 2020-2026 full run
timestamp = "20260618_130306"  # 2023-2026 full run

In [ ]:
import label_clusters

# Run the script (reads from BigQuery using run_timestamp)
label_clusters.main(timestamp)

Loading from BigQuery (run_timestamp=20260618_130306)...
  Classification: ocean-tech-adv-analytics-c-tfs.scope_drift_raw.classification_raw_20260618_130306
  Pub metadata:   ocean-tech-adv-analytics-c-tfs.scope_drift_raw.pub_metadata_raw_20260618_130306
  Cit links:      ocean-tech-adv-analytics-c-tfs.scope_drift_raw.cit_links_raw_20260618_130306

Processing macro level...
  Fetching up to 250 titles per cluster...
Downloading: 100%|██████████|
  Loaded 6,000 titles across 24 clusters

--- Labelling macro (24 clusters) ---
  [1/24] cluster 0 (1,237,462 papers) → Cancer Research
  [2/24] cluster 1 (319,941 papers) → Health Interventions


## Labelling with taxonomy 
- this is not as good as raw GPT so for now im going to drop it and ill ask toby what to do later

In [ ]:
import taxonomy_naming

# Run taxonomy naming (reads from BigQuery using run_timestamp)
taxonomy_naming.main(timestamp)

11:36:46 INFO     ============================================================
11:36:46 INFO     Taxonomy Naming Pipeline
11:36:46 INFO     Run timestamp: 20260618_130306
11:36:46 INFO     Cluster level: macro
11:36:46 INFO     ============================================================
11:36:46 INFO     Initializing clients...
11:36:50 INFO     Clients ready. Run date: 2026-07-10
11:36:50 INFO     Loading CWTS data from BigQuery...
c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
11:59:46 INFO     Loaded 3844339 papers from BigQuery
11:59:48 INFO     Sampling up to 300 papers per cluster...
11:59:48 INFO     Communities: 24 | Total publications (sampled): 7200
11:59:48 INFO     Loading taxonomy reference data...
11:59:53 INFO     L2 map: 27522 rows
c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_d

community_name
Cluster 0                                   Microbiome research
Cluster 1                                      Oncology nursing
Cluster 10                                       Food chemistry
Cluster 11                                     Machine learning
Cluster 12                           Cancer and Blood Disorders
Cluster 13                                           Cardiology
Cluster 14                                          Dermatology
Cluster 15                                  Reproductive Health
Cluster 16           Complementary and Rehabilitation Therapies
Cluster 17                              Comprehensive dentistry
Cluster 18                                          Haematology
Cluster 19                                        Ophthalmology
Cluster 2                       Comprehensive materials science
Cluster 20                   Sustainable and circular materials
Cluster 21                                              Surgery
Cluster 22               

,community_id,community_name,cluster_level,n_community_papers,tier,cluster_rank,cluster_key,cluster_name,taxonomy_level,match_mode,llm_confidence,llm_rationale,llm_reasoning,run_date
0,0,Cluster 0,macro,300,core,1,l1_583,Microbiome research,L1,combination,high,Microbiome research is a core focus due to its...,The publication community is primarily focused...,2026-07-10
1,0,Cluster 0,macro,300,core,2,l1_698,Oncology,L1,combination,high,"Oncology is a core focus with 26 articles, cov...",The publication community is primarily focused...,2026-07-10
2,0,Cluster 0,macro,300,core,3,l1_85,Medical genetics and genomics,L1,combination,high,Medical genetics and genomics is a core area w...,The publication community is primarily focused...,2026-07-10
3,0,Cluster 0,macro,300,core,4,l1_792,Immunopharmacology,L1,combination,high,Immunopharmacology is a core focus with 22 art...,The publication community is primarily focused...,2026-07-10
4,0,Cluster 0,macro,300,bleed,1,l1_794,Natural product pharmacology,L1,combination,high,Natural product pharmacology is a secondary fo...,The publication community is primarily focused...,2026-07-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,23,Cluster 23,macro,300,core,1,l1_688,Endocrinology and diabetes,L1,combination,high,The largest number of articles (99) are focuse...,The community's primary focus is on endocrinol...,2026-07-10
122,23,Cluster 23,macro,300,core,2,l1_698,Oncology,L1,combination,high,A significant number of articles (73) are rela...,The community's primary focus is on endocrinol...,2026-07-10
123,23,Cluster 23,macro,300,bleed,1,l1_702,Pathology,L1,combination,high,Pathology is relevant due to its focus on endo...,The community's primary focus is on endocrinol...,2026-07-10
124,23,Cluster 23,macro,300,bleed,2,l1_711,Surgery,L1,combination,high,"Surgery, particularly endocrine and head and n...",The community's primary focus is on endocrinol...,2026-07-10


### looking at scope

In [ ]:
import importlib
import journal_scope
from pathlib import Path

# Override config variables
journal_scope.SCOPE_LEVEL = "macro"
journal_scope.SCOPE_THRESHOLD = 0.80
journal_scope.MIN_PAPERS = 50
journal_scope.USE_GPT = False
journal_scope.OUTPUT_DIR = Path("cwts_output")

journal_scope.TARGET_JOURNALS = [
    "Frontiers in Immunology",
    "Frontiers in Public Health",
    "Frontiers in Medicine",
    "Frontiers in Oncology",
    "Frontiers in Psychology",
]

# Run
journal_scope.main()

Loading classification...
Loading metadata (journal + title)...
Merged: 67,823 publications across 5 journals
After MIN_PAPERS=50 filter: 5 journals, 67,823 papers

Computing scope at 'macro' level (threshold=80%)...
  Frontiers in Immunology                  n=19,501  core_clusters=  3  OOS=17.4%
  Frontiers in Medicine                    n=10,614  core_clusters=  9  OOS=18.4%
  Frontiers in Oncology                    n=12,794  core_clusters=  2  OOS=19.2%
  Frontiers in Psychology                  n=11,438  core_clusters=  3  OOS=19.9%
  Frontiers in Public Health               n=13,476  core_clusters=  7  OOS=17.7%

Saved journal scope → cwts_output\journal_scope.csv  (5 journals)
Saved paper flags   → cwts_output\paper_oos_flags.csv  (67,823 papers)

── Summary ──────────────────────────────────────────────────────
Journals analysed:        5
Mean OOS rate:            18.5%
Median OOS rate:          18.4%
Journals with OOS > 20%:  0
Journals with OOS > 40%:  0

Top 10 highest OOS 

### Generate Dashboard

In [ ]:
import subprocess
import sys
import os

# --- Config ---
CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# Build the scope dashboard from local CWTS files
# Uses: cwts_output/classification.txt, cwts_output/pub_metadata.txt, cwts_output/cit_links.txt
# Outputs: output/scope_dashboard.html
# Metadata pulled from BigQuery using run_timestamp

env = os.environ.copy()
env["CLUSTER_LEVEL"] = CLUSTER_LEVEL
env["RUN_TIMESTAMP"] = timestamp  # Pass timestamp to fetch metadata from BigQuery

result = subprocess.run(
    [sys.executable, "scripts/build_dashboard_from_cwts.py"],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.returncode != 0:
    print("ERRORS:")
    print(result.stderr)
else:
    print(f"\nDashboard ready: output/scope_dashboard.html")



Dashboard ready: output/scope_dashboard.html


In [ ]:
import subprocess
import sys
import os

# --- Config ---
CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# Build the drift dashboard (JSD trends, heatmap, entropy changes)
# Compares current cluster distribution vs baseline (2018-2020)
# Outputs: output/drift_dashboard.html

env = os.environ.copy()
env["CLUSTER_LEVEL"] = CLUSTER_LEVEL
env["RUN_TIMESTAMP"] = timestamp  # Pass timestamp to fetch metadata from BigQuery

result = subprocess.run(
    [sys.executable, "scripts/build_drift_dashboard_from_cwts.py"],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.returncode != 0:
    print("ERRORS:")
    print(result.stderr)
else:
    print(f"\nDashboard ready: output/drift_dashboard.html")



Dashboard ready: output/drift_dashboard.html


In [ ]:
import subprocess
import sys
import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the cluster bubble dashboard
# # Shows clusters as bubbles positioned by citation relationships
# # Outputs: output/cluster_bubbles.html

# env = os.environ.copy()
# env["CLUSTER_LEVEL"] = CLUSTER_LEVEL

# result = subprocess.run(
#     [sys.executable, "scripts/build_cluster_bubbles_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/cluster_bubbles.html")

In [ ]:
# import subprocess
# import sys

# # Build the clusters hierarchy dashboard
# # Shows macro/meso/micro clusters with GPT labels
# # Outputs: output/clusters.html

# result = subprocess.run(
#     [sys.executable, "scripts/build_clusters_from_cwts.py"],
#     capture_output=True,
#     text=True,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/clusters.html")



Dashboard ready: output/clusters.html
